# Pack a repo into QR sheets

Run this on the **offline** machine. `pack()` returns `{sheet_number: html}` and
writes nothing to disk — the sheets exist only in memory until you photograph them.

Needs `pip install segno`.

In [ ]:
import pathlib, qrpack

ROOT = pathlib.Path("../..").resolve()          # the repo to send
CODE = [".py", ".sh", ".zsh", ".bash", ".toml", ".yaml", ".yml", ".md", ".txt"]

sheets = qrpack.pack(ROOT, include=CODE)
sheets.keys()

## Displaying them

Each value is a self-contained HTML fragment with a white background, so it renders
correctly in a dark-theme notebook.

**Inline** is fine for checking one looks right. For actually photographing, use the
links below: each opens a single sheet in a new browser tab (a `data:` URL — still
nothing written to disk), where you can fullscreen it so the QR codes get the screen
pixels they need. The notebook output area is usually too narrow to show a whole
sheet at 1:1, and anything that scales the symbols down costs you decodability.

In [ ]:
import base64
from IPython.display import HTML, display

def in_new_tab(html, n):
    doc = f"<html><body style='margin:0;background:#fff;display:flex;" \
          f"align-items:center;justify-content:center;height:100vh'>{html}</body></html>"
    b64 = base64.b64encode(doc.encode()).decode()
    return (f"<a href='data:text/html;base64,{b64}' target='_blank' "
            f"style='font:600 13px system-ui;padding-right:14px'>sheet {n}</a>")

display(HTML(sheets[1]))                                     # preview one inline

display(HTML("<div style='padding-top:12px;font:600 13px system-ui'>"
             "open fullscreen to photograph:</div>"
             + "".join(in_new_tab(sheets[n], n) for n in sorted(sheets))))

Photograph one sheet per screen: square-on, filling the frame, tapped to focus, no
glare. Order doesn't matter and duplicates are harmless — every frame carries its own
index and checksum. Check the first photo is sharp before shooting the rest.

Then on the receiving machine:

```bash
python3 agunpack.py decode ~/photos/transfer --out ~/recovered
```

Fewer photographs: `pack(..., chunk=1100)` on a 1440p display, plus `cols=6, rows=3`
on 4K. `scale` is screen pixels per QR module — below about 3 the display physically
cannot show the detail, and no camera recovers it.

`qrpack.collect(ROOT, include=CODE)` returns the file list alone if you want to check
what's included before packing. Credentials — `.env`, `vault/*`, `*token*`, keys — are
excluded before `include` is applied, so no include list can pull one onto the screen.